## tl;dr

`서울미디어대학원대학교`는 개방ID `1861960053`으로 이미 별도 식별되어 있다. 2025년 서울 강서구에서 재적학생 379명과 6개 학과가 확인되므로, 서울 마포구에서 모든 핵심 지표가 0인 미확인 ID `2475617289`의 후보로 볼 수 없다. `상명대학교 디지털미디어대학원` 후보는 개방ID `1400377094`의 2009년 단일 기록이다.

## Context & Methods

EDSS 0101 학교개황에서 세 ID의 지역·규모·존속시점을 비교하고, 0221 대학원 재적학생현황에서 서울미디어대학원대학교의 2025년 학과 구성을 확인한다.

### Key Assumptions

- 개방ID는 문자열로 읽는다.
- 0은 결측치와 구분된 명시적 값으로 해석한다.
- 학교명 연결은 `data/processed/edss_0101_kedi_openid_identity_2009_2025.csv`의 다년도 교차검증 결과를 따른다.

## Data

In [1]:
import os
from pathlib import Path
import duckdb

database_path = Path(os.environ.get(
    'EDSS_DUCKDB_PATH',
    '/Users/joocheol/Documents/GitHub/edss/data/processed/edss/restricted/edss_all.duckdb',
))
assert database_path.exists(), f'DuckDB not found: {database_path}'
connection = duckdb.connect(str(database_path), read_only=True)
candidate_ids = ['2475617289', '1861960053', '1400377094']
database_path.name, duckdb.__version__

('edss_all.duckdb', '1.4.1')

## Results

### 1. ID별 지역과 학교 규모 비교

In [2]:
comparison_sql = '''
SELECT 조사년도, 개방ID, 시도명, 지역명, 본분교명, 학제유형명,
       고등교육학교_재적학생수, 고등교육학교_학과수, 고등교육학교_교원수
FROM higher_education.panel_0101
WHERE 개방ID IN ('2475617289', '1861960053', '1400377094')
  AND 조사년도 IN ('2009', '2010', '2025')
ORDER BY 조사년도, 개방ID
'''
comparison_rows = connection.execute(comparison_sql).fetchall()
comparison_columns = [item[0] for item in connection.description]
comparison_columns, comparison_rows

(['조사년도',
  '개방ID',
  '시도명',
  '지역명',
  '본분교명',
  '학제유형명',
  '고등교육학교_재적학생수',
  '고등교육학교_학과수',
  '고등교육학교_교원수'],
 [('2009', '1400377094', '서울', '서울 종로구', '본교', '대학원', '0', '0', '0'),
  ('2009', '1861960053', '서울', '서울 강서구', '본교', '대학원', '57', '2', '11'),
  ('2009', '2475617289', '서울', '서울 마포구', '본교', '대학원', '0', '0', '0'),
  ('2010', '1861960053', '서울', '서울 강서구', '본교', '대학원', '112', '3', '19'),
  ('2010', '2475617289', '서울', '서울 마포구', '본교', '대학원', '0', '0', '0'),
  ('2025', '1861960053', '서울', '서울 강서구', '본교(제1캠퍼스)', '대학원', '379', '6', '13'),
  ('2025', '2475617289', '서울', '서울 마포구', '본교(제1캠퍼스)', '대학원', '0', '0', '0')])

### 2. 서울미디어대학원대학교의 2025년 학과 구성

In [3]:
program_sql = '''
SELECT 학과명,
       SUM(TRY_CAST(재적학생_대학원_재적학생수 AS INTEGER)) AS 재적학생수,
       SUM(TRY_CAST(재적학생_대학원_재학생수 AS INTEGER)) AS 재학생수,
       SUM(TRY_CAST(재적학생_대학원_휴학생수 AS INTEGER)) AS 휴학생수
FROM higher_education.panel_0221
WHERE 개방ID = '1861960053' AND 조사년도 = '2025'
GROUP BY 학과명
ORDER BY 학과명
'''
program_rows = connection.execute(program_sql).fetchall()
program_columns = [item[0] for item in connection.description]
program_columns, program_rows

(['학과명', '재적학생수', '재학생수', '휴학생수'],
 [('AI스타트업학과', 5, 5, 0),
  ('e-비즈니스학과', 47, 46, 1),
  ('미디어비즈니스학과', 160, 138, 22),
  ('융합미디어학과', 8, 0, 8),
  ('융합예술디자인학과', 36, 32, 4),
  ('인공지능 응용소프트웨어학과', 123, 112, 11)])

### 3. 합계 일치 확인

In [4]:
program_total = sum(row[1] for row in program_rows)
school_total = connection.execute(
    "SELECT TRY_CAST(고등교육학교_재적학생수 AS INTEGER) FROM higher_education.panel_0101 WHERE 개방ID='1861960053' AND 조사년도='2025'"
).fetchone()[0]
assert program_total == school_total == 379
{'program_total': program_total, 'school_total': school_total, 'match': True}

{'program_total': 379, 'school_total': 379, 'match': True}

## Takeaways

- 서울미디어대학원대학교는 별도 개방ID `1861960053`으로 2009~2025년에 연결된다. 2009~2015년 교차표 명칭은 `한독미디어대학원대학교`, 2016년 이후는 `서울미디어대학원대학교`이다.
- 2025년에는 서울 강서구, 재적학생 379명, 6개 학과, 교원 13명으로 비영 지표가 명확하다.
- 미확인 ID `2475617289`는 같은 해 서울 마포구이며 재적학생·학과·교원이 모두 0이다. 지역과 규모가 모두 달라 동일 학교 후보에서 제외한다.
- `상명대학교 디지털미디어대학원` 후보는 종로구의 2009년 단일·0값 기록 `1400377094`이므로, 2009~2025년에 지속된 `2475617289`와도 일치하지 않는다.